# Day 4-03｜籃球軌跡追蹤與出手點估計
> Python 籃球運動資料分析課程  
> 以學生完成轉檔的影片建立球軌跡，並用速度變化估計出手 frame。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 從已轉檔影片讀取影格資料。
- 取得籃球中心點的 frame-by-frame 座標。
- 依據球軌跡與速度變化估計出手 frame。

## 完成產出
- 球軌跡 CSV 與軌跡圖。

## 課堂要求
- 先完成 Day 4-01 的影片上傳與轉檔。
- 按照本單元順序執行各段程式。
- 完成指定輸出後，記錄結果並供課堂討論。


## 課程流程
1. 讀取 Day 4-01 產生的 converted 影片。
2. 執行顏色式基準追蹤。
3. 儲存軌跡並判讀出手點。


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "src" / "course_setup.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("找不到 src/course_setup.py，請確認目前目錄位於課程 repo 內。")

from src.course_setup import DEFAULT_COURSE_ROOT, bootstrap_course_notebook  # noqa: E402

COURSE_ROOT = bootstrap_course_notebook(DEFAULT_COURSE_ROOT)


In [ ]:
import pandas as pd

from src.video_utils import pick_first_converted_video
from src.shooting_utils import track_orange_ball, estimate_release_frame
from src.plot_utils import plot_ball_path

try:
    video_path = pick_first_converted_video(COURSE_ROOT)
    HAS_VIDEO = True
    print("video_path:", video_path)
except FileNotFoundError as exc:
    video_path = None
    HAS_VIDEO = False
    print(exc)


In [ ]:
if HAS_VIDEO and video_path is not None:
    ball_df = track_orange_ball(video_path, max_frames=180)
else:
    ball_df = pd.DataFrame(columns=["frame", "x", "y", "area", "vx", "vy", "speed"])

print("tracked points:", len(ball_df))
ball_df.head()


In [ ]:
out_csv = COURSE_ROOT / "assets" / "results" / "d4_03_ball_track.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
ball_df.to_csv(out_csv, index=False)
print("saved:", out_csv)

if not ball_df.empty:
    plot_ball_path(
        ball_df, output_path=COURSE_ROOT / "assets" / "results" / "d4_03_ball_path.png"
    )
    release_frame = estimate_release_frame(ball_df)
else:
    release_frame = None
    print("沒有 converted 影片，略過球軌跡圖。")

print("estimated release frame:", release_frame)


## 模型選擇

本單元的 `track_orange_ball` 是顏色式基準方法，只適合用來說明「逐影格取得球中心點」的資料格式。在正式專案中，籃球追蹤建議分成兩層：

- 偵測模型：使用 Ultralytics YOLO 類型的 object detector，類別至少包含 `ball`；若任務需要判斷進球，可另外標註 `ball-in-basket`、rim 或 backboard。
- 時序追蹤：將每一 frame 的 ball detection 交給 ByteTrack 或 BoT-SORT 進行關聯；短暫漏偵可再用插值補齊。

球在籃球影片中通常很小，且容易被手部或身體遮擋。訓練資料應包含不同拍攝角度、球衣顏色、場地光線與壓縮品質，並優先檢查高解析度 frame 的偵測效果。
